In [3]:
# Installing the Pinecone connection client
!pip uninstall -y pinecone-client pinecone
!pip install pinecone

Found existing installation: pinecone-client 6.0.0
Uninstalling pinecone-client-6.0.0:
  Successfully uninstalled pinecone-client-6.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 41.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [pinecone]5/6 [pinecone]


In [4]:
import time
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

print("=== INITIALIZING PINECONE CLOUD INDEX OVER API PROXY ===")

# 1. Authentic Connection Setup
# Replace "YOUR_API_KEY" with the token provided from your dashboard console
PINECONE_API_KEY = "<Enter your APIs>" 
pc = Pinecone(api_key=PINECONE_API_KEY)

# 2. Define the Index Configuration to align with our $315 Token Budget
index_name = "infra-cost-index"
VECTOR_DIMENSIONS = 384  # Must perfectly match our SentenceTransformer model dimension!

# 3. Provision the Cloud Serverless Index if it doesn't already exist
if index_name not in pc.list_indexes().names():
    print(f"Creating a fresh index named '{index_name}'...")
    pc.create_index(
        name=index_name,
        dimension=VECTOR_DIMENSIONS,
        metric="cosine", # Using Cosine Similarity as our mathematical distance engine
        spec=ServerlessSpec(
            cloud="aws",      # Runs inside the same cloud provider as our t3.2xlarge cluster
            region="us-east-1"
        )
    )
    # Wait for the cloud node provisioning to complete
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)

# Connect directly to our active index endpoint
index = pc.Index(index_name)
print(f"Connected to Pinecone Index: '{index_name}' successfully.")

# 4. Initialize our Local CPU Embedding Model
model = SentenceTransformer('all-MiniLM-L6-v2')

# 5. Prepare our Data payload mapped directly to our Infrastructure Table
raw_text_data = "Our persistent storage allocation is 200 GB General Purpose EBS costing $16.00."
print(f"\nProcessing Ingestion Payload:\n -> '{raw_text_data}'")

# Generate the raw 384 decimals math array
vector_embeddings = model.encode(raw_text_data).tolist()

# 6. Execute the Upsert Operation (Upload the Vector + Text Metadata)
# Pinecone requires a unique ID string, the vector list, and optional human text metadata
payload = [
    (
        "vector_id_001", 
        vector_embeddings, 
        {"raw_text": raw_text_data, "layer": "Persistent Storage"}
    )
]

print("\nStreaming vector data across the API Gateway Proxy...")
upsert_response = index.upsert(vectors=payload)

print("\n=== PINECONE CLOUD STORAGE METADATA ===")
print(f"Upsert Response Status:     {upsert_response}")
print(f"Total Records Confirmed:    {upsert_response['upserted_count']}")
print("=========================================================")
print("SUCCESS: Vector and Metadata are safely secured in Pinecone Cloud!")

=== INITIALIZING PINECONE CLOUD INDEX OVER API PROXY ===
Creating a fresh index named 'infra-cost-index'...
Connected to Pinecone Index: 'infra-cost-index' successfully.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Processing Ingestion Payload:
 -> 'Our persistent storage allocation is 200 GB General Purpose EBS costing $16.00.'

Streaming vector data across the API Gateway Proxy...

=== PINECONE CLOUD STORAGE METADATA ===
Upsert Response Status:     UpsertResponse(upserted_count=1)
Total Records Confirmed:    1
SUCCESS: Vector and Metadata are safely secured in Pinecone Cloud!
